# 20 — ICI Specificity of CIDER T-cell Score (Main Figure, TMB-Adjusted)

Main-figure companion to the full supplementary analysis in
`15_ici_specificity.ipynb`. Tests whether the CIDER T-cell response and
survival associations are specific to immune checkpoint inhibitor (ICI)
therapy, using the same `cider_log2 * is_ici` interaction design.

**Figure numbering note:** this notebook's main-text output is now Section
7's figure, "Predictive Specificity for ICI Therapy vs. General Prognostic
Effects" (`ici_specificity_predictive_vs_prognostic.rds`/`.png`), assembled
as **Figure 6**. Section 5's own forest plot (`main_ici_specificity_figure`)
was originally the main-text figure (formerly Figure 7) but has been
retired from the main figure sequence -- superseded by Section 7, its
updated version. Section 5's fitting/plotting code is kept as-is because
Section 6 (adjustment-sensitivity diagnostic) and Section 7 both reuse its
shared helpers (`label_df`, `GROUP_ORDER`, `ENDPOINT_LEVELS`,
`adaptive_breaks()`, `results`, `parent_dfs`, `interaction_dfs`) -- see each
section's own markdown for what it still builds and why.

Section 5's original simplification choices, described below, are still
what every section in this notebook builds on:

- **Fewer interaction tests.** Only the main-cohort ICI-specificity tests are
  shown: ICI vs. each of three treatment classes (Chemotherapy, Targeted
  therapy, Hormonal therapy), ICI vs. Non-ICI Therapy overall, and ICI vs.
  Non-ICI Therapy restricted to microsatellite-stable (MSS) patients on both
  arms. The MSI-high and TMB-High/TMB-Low stratified sensitivity tests from
  the supplement are not repeated here -- they are robustness checks, not
  main-cohort tests (see `15_ici_specificity.ipynb`). "Multiple therapy"
  (pooled Chemo + Anti-HER2 / Chemo + Anti-VEGF combinations) is also
  dropped from this main-figure version -- it is the smallest and most
  heterogeneous of the four treatment classes, and its within-group CIDER
  effect size is unstable enough (wide CI, happens to land near the ICI
  estimate) that it reads as a false contradiction against its own
  significant interaction term. It remains in `15_ici_specificity.ipynb` for
  completeness.
- **No mechanism-level breakdown.** Per-group effect sizes are shown only at
  the top/mid level (ICI Therapy, Non-ICI Therapy, and the three remaining
  treatment classes) -- the individual mechanism sub-group rows (e.g.
  specific chemo regimens) are dropped for a cleaner main-text figure; they
  remain available in the supplement.

**Covariates:** `cohort_pan` (tumor type) + `scaled_age` + `sex_female` +
`biopsy_structure` + `scaled_purity` + **`scaled_tmb`** (log2(TMB per Mb + 1),
standardized -- same convention as `5_models.ipynb`/`9_roc.ipynb`). The only
methodological difference from `15_ici_specificity.ipynb` is this last
covariate: TMB is a correlated but distinct genomic correlate of ICI
response, and rather than only testing it via TMB-High/TMB-Low stratified
sub-cohorts (as the supplement does), this main-figure version adjusts for
it directly in every model, so the headline specificity test already
accounts for TMB. Age and purity are each standardized once across this
notebook's full analysis population (all ICI + comparator patients pooled),
mirroring `5_models.ipynb`'s single-pooled-scaling convention; a covariate is
dropped from a given model only if it's constant within that fitted group
(same "drop if constant" pattern as `15_ici_specificity.ipynb`).

**Caution on purity:** as in `15_ici_specificity.ipynb` -- tumor purity
partly reflects T-cell infiltration itself (the non-tumor fraction PURPLE
measures includes infiltrating T cells), so a weaker interaction term after
purity adjustment is not automatically evidence against specificity.

**Note for readers comparing to Figures 3/4:** because this figure adjusts
for TMB (and the headline Figures 3/4 do not), its within-group CIDER effect
sizes for "ICI Therapy" and "Non-ICI Therapy" will not exactly match the
unadjusted estimates shown there -- this is an intentional TMB-adjusted
sensitivity view of the same data, not a correction to the main results.

**Row structure:**
- **ICI Specificity** (leading panel, diamond markers, labelled "ICI vs. X"):
  five `cider_log2 * is_ici` interaction models, each fit on the pooled ICI +
  one comparator's patients (the three remaining treatment classes, Non-ICI
  Therapy overall, and Non-ICI Therapy restricted to MSS patients on both
  arms). The plotted value is the interaction term (`cider_log2:is_ici`), a
  ratio-of-ORs (BOR/DCB) or ratio-of-HRs (PFS/OS) -- how much stronger (or
  weaker) the CIDER-score effect is in ICI patients versus that comparator.
  This is the direct test of "specificity": the per-group rows below show
  the effect size *within* each group, but only the interaction term
  formally tests whether that effect *differs* between ICI and the
  comparator.
- Per-group rows (circles = ICI, triangles = Non-ICI): the within-group
  CIDER effect size for ICI Therapy, Non-ICI Therapy, and each of the three
  remaining treatment classes.

**Layout:** `facet_grid(section ~ endpoint)` -- ICI Therapy, Non-ICI Therapy,
and the three remaining treatment classes as separate row panels; BOR, DCB,
PFS, OS as column panels.

CIs use the Wald method (±1.96 × SE on the log scale). Tumor-type covariate
cells with < 20 patients are recoded to 'Other' (all patients retained).

FDR correction (BH) is applied within each endpoint, across all tests
fitted for that endpoint (per-group effect-size tests and the five
ICI-specificity interaction tests together, as one family).

**Input:** `ready.rds`, `READY_DIR/clinical_update.txt`
**Output:** `figs/main_ici_specificity_figure.rds`/`.png` (Section 5 -- retired from the
main figure sequence, kept only as the shared fitting/labeling step Sections 6-7 build
on) **+** `figs/main_ici_specificity_figure_adjustment_sensitivity.rds`/`.png`
(Section 6 -- diagnostic alternative, not part of the submission figure set) **+**
`figs/ici_specificity_predictive_vs_prognostic.rds`/`.png` (Section 7 -- **Figure 6**,
the submission figure)


In [ ]:
library(dplyr)
library(tidyr)
library(purrr)
library(broom)
library(survival)
library(ggtext)
library(ggplot2)
library(data.table)


In [ ]:
args <- commandArgs(trailingOnly = FALSE)
script_path <- sub("--file=", "", args[grep("--file=", args)])
SCRIPT_DIR <- if (length(script_path) > 0) {
  normalizePath(dirname(script_path), mustWork = FALSE)
} else {
  getwd()
}

source(file.path(SCRIPT_DIR, "../helpers/map.r"))
source(file.path(SCRIPT_DIR, "../helpers/shortcuts.r"))
source(file.path(SCRIPT_DIR, "../helpers/config.r"))   # ICI_TREATMENT_NORM, ICI_MECHANISM_MAP
source(file.path(SCRIPT_DIR, "../helpers/themes.r"))


## 1 — Data preparation


In [ ]:
# Load clinical metadata.
# 'treatment_str' is renamed from 'treatment' to avoid collision with the
# existing 'treatment' column in ready.rds after the join.
clinical_raw <- data.table::fread(paste0(READY_DIR, "clinical_update.txt")) |>
  dplyr::transmute(
    sampleId       = sampleId,
    treatment_type = type,
    treatment_str  = treatment,
    mechanism_raw  = mechanism
  ) |>
  dplyr::distinct(sampleId, .keep_all = TRUE)

message("Treatment type distribution:")
print(sort(table(clinical_raw$treatment_type), decreasing = TRUE))


In [ ]:
# Load full ready object (T-cell group); join clinical metadata.
# ready.rds duplicates every patient: once with the natural cohort_broad and
# once with cohort_broad = 'ICI-Treated' (see 4_base.ipynb bind_rows call).
# Deduplicate by sampleId immediately to avoid inflated Ns.
# Classify non-ICI treatment regimens into mechanism sub-groups based on the
# treatment string. Only used here to DEFINE the Targeted-therapy/
# Multiple-therapy parent groups below (e.g. "CDK4/6i" -> Targeted therapy,
# "Chemo + Anti-HER2/VEGF" -> Multiple therapy) -- unlike
# 15_ici_specificity.ipynb, this main-figure version does not go on to build
# individual mechanism sub-group rows from this classification.
classify_non_ici_mech <- function(trt) {
  if (is.na(trt) || trt == "") return(NA_character_)
  t <- tolower(trt)
  has <- function(...) any(vapply(c(...), function(d) grepl(d, t, fixed = TRUE), logical(1)))
  # Exclude ICI-containing regimens from non-ICI sections
  if (has("pembrolizumab","nivolumab","atezolizumab","ipilimumab",
           "durvalumab","avelumab","tremelimumab")) return(NA_character_)
  # Targeted / biologic add-ons
  if (has("palbociclib","ribociclib","abemaciclib"))              return("CDK4/6i")
  if (has("trastuzumab","pertuzumab","lapatinib","ado-trastuzumab")) return("Chemo + Anti-HER2")
  if (has("bevacizumab","ramucirumab"))                           return("Chemo + Anti-VEGF")
  # Cetuximab/panitumumab: too few responders to model — drop
  if (has("cetuximab","panitumumab"))                             return(NA_character_)
  # Pure chemotherapy sub-groups (ordered from most specific to most general)
  has_plat <- grepl("carboplatin|cisplatin|oxaliplatin", t)
  has_gem  <- grepl("gemcitabine", t)
  has_tax  <- grepl("paclitaxel|docetaxel|cabazitaxel|nab-paclitaxel", t)
  has_fp   <- grepl("capecitabine|fluorouracil|trifluridine|tipiracil", t)
  has_anth <- grepl("doxorubicin|epirubicin", t)
  has_iri  <- grepl("irinotecan", t)
  has_etop <- grepl("etoposide", t)
  if (has_gem  && has_plat && !has_tax && !has_fp)  return("Gemcitabine + Platinum")
  if (has_tax  && has_plat)                         return("Taxane + Platinum")
  if (has_fp   && grepl("oxaliplatin", t) && !has_tax) return("Fluoropyrimidine + Oxaliplatin")
  # Irinotecan alone or with FP: too few responders — drop
  if (has_iri)                                      return(NA_character_)
  if (has_tax  && !has_plat)                        return("Taxane")
  if (has_fp   && !has_plat && !has_anth)           return("Fluoropyrimidine")
  if (has_anth)                                     return("Anthracycline")
  if (has_etop && has_plat)                         return("Etoposide + Platinum")
  if (has_plat && !has_gem && !has_tax && !has_fp)  return("Platinum")
  return(NA_character_)
}

ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds")) |>
  dplyr::filter(gp == "T-cell") |>
  dplyr::distinct(sampleId, .keep_all = TRUE) |>
  dplyr::left_join(clinical_raw, by = "sampleId") |>
  dplyr::mutate(
    cider_log2 = log2(cider_score_depth_adj + 1),
    response   = as.integer(response),
    dcb        = as.integer(dcb),
    os_months  = osDays / 30.44,
    pfs_months = pfsEventDays / 30.44,
    cohort_pan = as.character(cohort_pan),
    # Adjustment covariates: age, sex, biopsy location, tumor purity, and
    # TMB. Age, purity, and TMB are each standardized once across this
    # notebook's full analysis population (all ICI + comparator patients
    # pooled), mirroring 5_models.ipynb's single-pooled-scaling convention
    # for covariates that are adjustment terms only (not effects of
    # interest in their own right). scaled_tmb uses the same
    # log2(tmbPerMb + 1) + scale() convention as 5_models.ipynb/9_roc.ipynb,
    # and is the one covariate this notebook adds beyond
    # 15_ici_specificity.ipynb (see notebook intro). biopsyStructure
    # arrives from 3_meta.ipynb already collapsed to BIOPSY_STRUCTURE_LEVELS
    # (helpers/config.r) -- reused as-is here rather than re-grouped, since
    # that collapsing already exists specifically to avoid small
    # biopsy-location cells. See this notebook's intro for a caution on
    # purity specifically -- it's partly a consequence of T-cell
    # infiltration, not purely an independent confounder of it.
    scaled_age       = as.vector(scale(as.numeric(age))),
    sex_female       = as.integer(tolower(sex) %in% c("female", "f")),
    biopsy_structure = biopsyStructure,
    scaled_purity    = as.vector(scale(as.numeric(purity))),
    scaled_tmb       = as.vector(scale(log2(tmbPerMb + 1)))
  ) |>
  dplyr::mutate(
    treatment_norm = unname(ICI_TREATMENT_NORM[treatment_str]),
    treatment_norm = dplyr::coalesce(treatment_norm, treatment_str)
  ) |>
  dplyr::mutate(
    ici_mechanism   = unname(ICI_MECHANISM_MAP[treatment_norm]),
    ici_mechanism   = dplyr::coalesce(ici_mechanism, "Other"),
    # ICI patients: use ICI mechanism map.
    # Chemo/Multiple therapy: reclassify by treatment string.
    # Targeted/Hormonal therapy: keep original mechanism_raw label.
    mechanism_group = dplyr::case_when(
      cpi == TRUE ~ ici_mechanism,
      treatment_type %in% c("Chemotherapy", "Multiple therapy") ~
        unname(vapply(treatment_str, classify_non_ici_mech, character(1))),
      TRUE ~ mechanism_raw
    )
  ) |>
  dplyr::select(-treatment_norm, -ici_mechanism) |>
  # Restrict to patients with both response (BOR) and OS annotation,
  # matching the inclusion criteria used in the main analyses.
  dplyr::filter(!is.na(response), !is.na(os_months), !is.na(osEvent))

message("Total unique patients: ", dplyr::n_distinct(ready$sampleId))


## 2 — Parent group definitions


In [ ]:
# Top-level and mid-level treatment-class groups, used both for the
# per-group effect-size rows below and (via INTERACTION_COMPARATORS) for the
# ICI-specificity interaction tests. Unlike 15_ici_specificity.ipynb, this
# main-figure version stops here -- it does not derive individual mechanism
# sub-groups within each class, since those rows are dropped from the main
# figure (see notebook intro).
# "Multiple therapy" (pooled Chemo + Anti-HER2 / Chemo + Anti-VEGF
# combinations) is intentionally excluded here -- it is the smallest and
# most heterogeneous of the treatment classes in 15_ici_specificity.ipynb,
# and its within-group CIDER effect size is unstable enough to read as a
# false contradiction against its own significant interaction term (see
# notebook intro). It remains in the full supplement.
parent_dfs <- list(
  "ICI Therapy"      = ready |> dplyr::filter(cpi == TRUE) |>
                        dplyr::mutate(cohort_pan = as.character(cohort_broad)),
  "Non-ICI Therapy"  = ready |> dplyr::filter(cpi == FALSE),
  "Chemotherapy"     = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Chemotherapy"),
  "Targeted therapy" = ready |> dplyr::filter(cpi == FALSE,
                          treatment_type == "Targeted therapy" | mechanism_group == "CDK4/6i"),
  "Hormonal therapy" = ready |> dplyr::filter(cpi == FALSE, treatment_type == "Hormonal therapy")
)

message("Parent group sizes:")
print(purrr::imap_dfr(parent_dfs, ~ dplyr::tibble(group = .y, n = nrow(.x))))


## 3 — Regression functions


In [ ]:
MIN_N        <- 20
MIN_EVENTS   <- 5
# cohort_pan levels with fewer than MIN_COHORT_N patients are recoded to
# 'Other' so all patients are retained while sparse covariate cells are
# collapsed, reducing separation risk in small sub-groups.
MIN_COHORT_N <- 10

# 95% Wald CI helper: avoids profile.glm() convergence failures.
wald_ci_exp <- function(fit, term_name) {
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == term_name) |>
    dplyr::mutate(
      conf.low  = exp(estimate - 1.96 * std.error),
      conf.high = exp(estimate + 1.96 * std.error),
      estimate  = exp(estimate)
    )
}

# Collapse small cohort_pan cells into 'Other' (retains all patients).
recode_small_cohorts <- function(df) {
  df |>
    dplyr::group_by(cohort_pan) |>
    dplyr::mutate(
      cohort_pan = ifelse(dplyr::n() < MIN_COHORT_N, "Other", cohort_pan)
    ) |>
    dplyr::ungroup()
}

# Extra adjustment covariates (age, sex, biopsy location, purity, TMB), added
# on top of cohort_pan. Each is included only if it actually varies within
# the fitted group -- same "drop if constant" pattern already used for
# cohort_pan above -- since a constant covariate in a small comparator arm
# would otherwise waste a degree of freedom or block convergence for no
# informational gain. scaled_tmb is the one addition versus
# 15_ici_specificity.ipynb (see notebook intro).
# `exclude` drops one or more covariates from the returned set regardless of
# whether they vary -- used by build_adjustment_covars() below for the
# "Adjusted (no purity)" spec.
extra_adjustment_covariates <- function(df, exclude = character(0)) {
  extra <- character(0)
  if (dplyr::n_distinct(df$scaled_age, na.rm = TRUE) > 1)       extra <- c(extra, "scaled_age")
  if (dplyr::n_distinct(df$sex_female, na.rm = TRUE) > 1)       extra <- c(extra, "sex_female")
  if (dplyr::n_distinct(df$biopsy_structure, na.rm = TRUE) > 1) extra <- c(extra, "biopsy_structure")
  if (dplyr::n_distinct(df$scaled_purity, na.rm = TRUE) > 1)    extra <- c(extra, "scaled_purity")
  if (dplyr::n_distinct(df$scaled_tmb, na.rm = TRUE) > 1)       extra <- c(extra, "scaled_tmb")
  setdiff(extra, exclude)
}

# Named covariate-adjustment specs, shared by every fit_* function below via
# the `adjustment` argument (default "Adjusted" everywhere, so this
# refactor does not change any of Section 4's primary, Figure-7-feeding
# results). "Unadjusted" adds nothing at all -- not even cohort_pan -- same
# convention as 5_models.ipynb's own "None" spec. "Adjusted" is this
# notebook's primary spec. "Adjusted (no purity)" is the same minus
# scaled_purity, used by Section 4b's sensitivity check and Section 6's
# alternative figure below.
ADJUSTMENT_LEVELS <- c("Unadjusted", "Adjusted", "Adjusted (no purity)")

build_adjustment_covars <- function(df, adjustment) {
  switch(adjustment,
    "Unadjusted" = character(0),
    "Adjusted" = c(
      if (dplyr::n_distinct(df$cohort_pan) > 1) "cohort_pan",
      extra_adjustment_covariates(df)
    ),
    "Adjusted (no purity)" = c(
      if (dplyr::n_distinct(df$cohort_pan) > 1) "cohort_pan",
      extra_adjustment_covariates(df, exclude = "scaled_purity")
    ),
    stop("Unknown adjustment spec: ", adjustment)
  )
}

fit_logistic <- function(df, outcome_col, group_label, endpoint_label,
                         adjustment = "Adjusted") {
  # drop_na() always requires every adjustment-related column, regardless of
  # `adjustment` -- so the fitted population is IDENTICAL across all three
  # specs for a given group/endpoint, and any difference in the estimate is
  # attributable to the covariate set, not a different sample (same
  # principle as Section 4b's purity check).
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(outcome_col)) |>
    recode_small_cohorts()

  outcome <- df[[outcome_col]]
  if (nrow(df) < MIN_N || sum(outcome, na.rm = TRUE) < MIN_EVENTS ||
      length(unique(outcome)) < 2) {
    message("Skipping ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(outcome, na.rm = TRUE), ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars_lr <- c("cider_log2", build_adjustment_covars(df, adjustment))
  fit <- tryCatch(
    suppressWarnings(
      glm(reformulate(covars_lr, outcome_col), data = df, family = binomial())
    ),
    error = function(e) NULL
  )
  if (is.null(fit)) return(NULL)
  wald_ci_exp(fit, "cider_log2") |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(outcome, na.rm = TRUE),
                  model = "Logistic", adjustment = adjustment)
}

fit_cox <- function(df, time_col, event_col, group_label, endpoint_label,
                    adjustment = "Adjusted") {
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(c(time_col, event_col))) |>
    dplyr::filter(.data[[time_col]] > 0) |>
    recode_small_cohorts()

  events <- df[[event_col]]
  if (nrow(df) < MIN_N || sum(events, na.rm = TRUE) < MIN_EVENTS) {
    message("Skipping ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(events, na.rm = TRUE), ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars_cox <- paste(c("cider_log2", build_adjustment_covars(df, adjustment)), collapse = " + ")
  fmla <- as.formula(paste0("Surv(", time_col, ", ", event_col, ") ~ ", covars_cox))
  fit <- tryCatch(
    suppressWarnings(survival::coxph(fmla, data = df)),
    error = function(e) NULL
  )
  if (is.null(fit)) return(NULL)
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == "cider_log2") |>
    dplyr::mutate(
      conf.low  = exp(estimate - 1.96 * std.error),
      conf.high = exp(estimate + 1.96 * std.error),
      estimate  = exp(estimate)
    ) |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(events, na.rm = TRUE),
                  model = "Cox", adjustment = adjustment)
}

run_all_endpoints <- function(df, group_label, adjustment = "Adjusted") {
  dplyr::bind_rows(
    fit_logistic(df, "response",   group_label, "BOR", adjustment),
    fit_logistic(df, "dcb",        group_label, "DCB", adjustment),
    fit_cox(df, "os_months",  "osEvent",  group_label, "OS",  adjustment),
    fit_cox(df, "pfs_months", "pfsEvent", group_label, "PFS", adjustment)
  )
}

# ── Interaction models (ICI-specificity test) ──────────────────────────────────
# Fits cider_log2 * is_ici on the pooled {ICI Therapy} + {one comparator} data
# and extracts the cider_log2:is_ici term. This is a formal test of whether the
# CIDER-score effect differs between ICI and that comparator (specificity),
# as opposed to fit_logistic()/fit_cox() above which only estimate the effect
# *within* one group at a time.

fit_interaction_logistic <- function(df, outcome_col, group_label, endpoint_label,
                                     adjustment = "Adjusted") {
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, is_ici, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(outcome_col)) |>
    recode_small_cohorts()

  outcome <- df[[outcome_col]]
  if (nrow(df) < MIN_N || sum(outcome, na.rm = TRUE) < MIN_EVENTS ||
      length(unique(outcome)) < 2 || dplyr::n_distinct(df$is_ici) < 2) {
    message("Skipping interaction ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(outcome, na.rm = TRUE), ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars <- c("cider_log2 * is_ici", build_adjustment_covars(df, adjustment))
  fit <- tryCatch(
    suppressWarnings(glm(reformulate(covars, outcome_col), data = df, family = binomial())),
    error = function(e) NULL
  )
  if (is.null(fit)) return(NULL)
  wald_ci_exp(fit, "cider_log2:is_ici") |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(outcome, na.rm = TRUE),
                  model = "Logistic (interaction)", adjustment = adjustment)
}

fit_interaction_cox <- function(df, time_col, event_col, group_label, endpoint_label,
                                adjustment = "Adjusted") {
  df <- df |>
    tidyr::drop_na(cider_log2, cohort_pan, is_ici, scaled_age, sex_female, biopsy_structure,
                   scaled_purity, scaled_tmb, dplyr::all_of(c(time_col, event_col))) |>
    dplyr::filter(.data[[time_col]] > 0) |>
    recode_small_cohorts()

  events <- df[[event_col]]
  if (nrow(df) < MIN_N || sum(events, na.rm = TRUE) < MIN_EVENTS ||
      dplyr::n_distinct(df$is_ici) < 2) {
    message("Skipping interaction ", group_label, " / ", endpoint_label,
            " (n=", nrow(df), ", events=", sum(events, na.rm = TRUE), ", adjustment=", adjustment, ")")
    return(NULL)
  }
  covars <- paste(c("cider_log2 * is_ici", build_adjustment_covars(df, adjustment)), collapse = " + ")
  fmla <- as.formula(paste0("Surv(", time_col, ", ", event_col, ") ~ ", covars))
  fit <- tryCatch(suppressWarnings(survival::coxph(fmla, data = df)), error = function(e) NULL)
  if (is.null(fit)) return(NULL)
  broom::tidy(fit, conf.int = FALSE, exponentiate = FALSE) |>
    dplyr::filter(term == "cider_log2:is_ici") |>
    dplyr::mutate(
      conf.low  = exp(estimate - 1.96 * std.error),
      conf.high = exp(estimate + 1.96 * std.error),
      estimate  = exp(estimate)
    ) |>
    dplyr::mutate(group = group_label, endpoint = endpoint_label,
                  n = nrow(df), n_events = sum(events, na.rm = TRUE),
                  model = "Cox (interaction)", adjustment = adjustment)
}

run_interaction_endpoints <- function(df, group_label, adjustment = "Adjusted") {
  dplyr::bind_rows(
    fit_interaction_logistic(df, "response",   group_label, "BOR", adjustment),
    fit_interaction_logistic(df, "dcb",        group_label, "DCB", adjustment),
    fit_interaction_cox(df, "os_months",  "osEvent",  group_label, "OS",  adjustment),
    fit_interaction_cox(df, "pfs_months", "pfsEvent", group_label, "PFS", adjustment)
  )
}


## 4 — Fit models


In [ ]:
parent_results <- purrr::imap(
  parent_dfs,
  ~ run_all_endpoints(.x, .y)
) |> dplyr::bind_rows() |>
  dplyr::mutate(
    group_level = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), "top", "mid"),
    parent      = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), NA_character_, "Non-ICI Therapy")
  )

# ── Interaction models (ICI-specificity test) ──────────────────────────────────
# Pool ICI Therapy with each comparator in turn (the three named treatment
# classes, plus Non-ICI Therapy overall as a coarser summary test) and fit
# cider_log2 * is_ici; the interaction term tests whether the CIDER-score
# effect genuinely differs between ICI and that comparator, rather than just
# looking stronger in the ICI row above by eye. "Multiple therapy" is
# excluded from this main-figure version (see notebook intro and the
# parent_dfs definition above).
INTERACTION_COMPARATORS <- c("Non-ICI Therapy", "Chemotherapy", "Targeted therapy", "Hormonal therapy")

interaction_dfs <- purrr::map(
  INTERACTION_COMPARATORS,
  function(comparator) {
    dplyr::bind_rows(parent_dfs[["ICI Therapy"]], parent_dfs[[comparator]]) |>
      dplyr::mutate(is_ici = as.integer(cpi == TRUE))
  }
) |> stats::setNames(paste0(INTERACTION_COMPARATORS, "__interaction"))

# Sixth interaction test: the same ICI vs. Non-ICI Therapy comparison above,
# restricted to microsatellite-stable (MSS) patients on both arms. MSI-high
# tumors respond to ICI through a largely CIDER-independent mechanism (high
# neoantigen burden from mismatch-repair deficiency), so pooling MSI-high and
# MSS patients into the headline "Non-ICI Therapy" interaction test above
# could make the ICI arm's CIDER-score effect look stronger than it really is
# for ordinary (MSS) patients. `msi_high == 0` also drops patients with
# unknown MSI status (NA), rather than assuming they are MSS. (The mirror-image
# MSI-high test, and the TMB-High/TMB-Low stratified tests, are supplementary
# sensitivity checks only -- see 15_ici_specificity.ipynb -- since this
# main-figure version already adjusts for TMB directly in every model.)
interaction_dfs[["Non-ICI (MSS)__interaction"]] <-
  dplyr::bind_rows(
    parent_dfs[["ICI Therapy"]]     |> dplyr::filter(msi_high == 0),
    parent_dfs[["Non-ICI Therapy"]] |> dplyr::filter(msi_high == 0)
  ) |>
  dplyr::mutate(is_ici = as.integer(cpi == TRUE))

interaction_results <- purrr::imap(
  interaction_dfs,
  ~ run_interaction_endpoints(.x, .y)
) |> dplyr::bind_rows() |>
  dplyr::mutate(
    group_level = "interaction",
    parent      = sub("__interaction$", "", group)
  )

# Per-arm sample sizes (N = X ICI + N = Y comparator) for the row labels below,
# computed once from the pooled interaction data rather than per endpoint --
# the split barely moves with endpoint-specific missingness, and one stable
# pair of numbers per test is clearer than four slightly different ones.
interaction_n_df <- purrr::imap_dfr(
  interaction_dfs,
  function(df, group_name) {
    dplyr::tibble(
      group   = group_name,
      n_ici   = sum(df$is_ici == 1L),
      n_other = sum(df$is_ici == 0L)
    )
  }
)
interaction_results <- interaction_results |>
  dplyr::left_join(interaction_n_df, by = "group")

message("\nInteraction models fitted: ", nrow(interaction_results), " rows across ",
        length(interaction_dfs), " comparisons (", length(INTERACTION_COMPARATORS),
        " treatment-class comparators, plus Non-ICI Therapy restricted to MSS patients)")
# NOTE: "Multiple therapy" is intentionally excluded from this count -- see
# notebook intro and the parent_dfs definition above.

results <- dplyr::bind_rows(parent_results, interaction_results)

# ── FDR correction ────────────────────────────────────────────────────────────
# This figure fits a per-group effect-size test for each of 5 groups plus 5
# ICI-specificity interaction tests, per endpoint. Benjamini-Hochberg FDR
# correction is applied across all of them together within each endpoint
# (BOR, DCB, PFS, OS are treated as separate test families) — every model
# fitted for a given endpoint in this figure is corrected as one family,
# rather than splitting the interaction tests into their own family.
#
# Intervals are left as standard fixed-width 95% Wald CIs (a CI's width
# should reflect a pre-specified coverage probability, not be derived from
# that same test's own p-value — deriving width from p.adj degenerates to a
# near-zero-width interval for non-significant tests). FDR-adjusted p-values
# are used only to flag significance (stars / point opacity below).
results <- results |>
  dplyr::group_by(endpoint) |>
  dplyr::mutate(p.adj = p.adjust(p.value, method = "fdr")) |>
  dplyr::ungroup()

message("\nResults summary:")
print(results |> dplyr::select(group, group_level, endpoint, n, n_events,
                                estimate, conf.low, conf.high, p.value, p.adj))


## 4b — Sensitivity check: does purity adjustment change the interaction results?

**Diagnostic only** -- does not change the primary, purity-adjusted model
that `p_ici_specificity_main` is built from (Section 5, retired from the main
figure sequence but still the shared fitting step Sections 6-7 build on).
This refits
only the five ICI-specificity interaction tests with `scaled_purity` dropped
from the covariate set (everything else -- cohort_pan, age, sex, biopsy
location, TMB -- unchanged), on the exact same patient population as the
purity-adjusted fit (the `drop_na()` requirement for `scaled_purity` is kept
even though the term itself is excluded, specifically so N is identical
between the two and any difference in the estimate is attributable to the
covariate, not a different sample).

This exists because tumor purity is not a clean, independent confounder
here -- lower purity partly reflects *more* immune infiltration, which is
exactly what CIDER itself measures (see notebook intro) -- so it is worth
seeing directly how much the purity term is attenuating the interaction
estimate, rather than only asserting that it might be. The comparison below
is not a second FDR-corrected test family; it is a sensitivity check on the
primary model's estimates.


In [ ]:
# Refit the same five interaction comparisons (interaction_dfs, defined above)
# with the "Adjusted (no purity)" spec (ADJUSTMENT_LEVELS, defined above).
interaction_results_no_purity <- purrr::imap(
  interaction_dfs,
  ~ run_interaction_endpoints(.x, .y, adjustment = "Adjusted (no purity)")
) |> dplyr::bind_rows()

purity_sensitivity <- interaction_results |>
  dplyr::select(group, endpoint, n, n_events,
                estimate_purity_adj  = estimate,
                conf.low_purity_adj  = conf.low,
                conf.high_purity_adj = conf.high,
                p_purity_adj         = p.value) |>
  dplyr::inner_join(
    interaction_results_no_purity |>
      dplyr::select(group, endpoint,
                    estimate_no_purity  = estimate,
                    conf.low_no_purity  = conf.low,
                    conf.high_no_purity = conf.high,
                    p_no_purity         = p.value),
    by = c("group", "endpoint")
  ) |>
  # Positive pct_shift = purity adjustment moved the estimate AWAY from 1
  # (stronger interaction); negative = moved it TOWARD 1 (attenuated), which
  # is the direction the notebook intro's caution anticipates.
  dplyr::mutate(
    pct_shift = 100 * (abs(log(estimate_purity_adj)) - abs(log(estimate_no_purity))) /
                        abs(log(estimate_no_purity))
  ) |>
  dplyr::arrange(endpoint, group)

message("\n== Sensitivity check: purity-adjusted vs. purity-unadjusted ICI-specificity interaction terms ==")
message("Same population in both columns (drop_na() still requires non-missing purity even when the")
message("term is excluded from the model) -- any difference reflects the covariate, not a sample shift.")
message("pct_shift: negative = purity adjustment attenuated the interaction effect toward 1 (the")
message("expected direction per the notebook intro's purity caution); positive = it strengthened it.")
print(purity_sensitivity, n = Inf)


## 5 — Forest plot


In [ ]:
# ── Label lookup ─────────────────────────────────────────────────────────────

group_n <- results |>
  dplyr::group_by(group, group_level, parent) |>
  dplyr::summarise(
    n_total = max(n, na.rm = TRUE),
    # NA for every non-interaction group (n_ici/n_other only exist on
    # interaction_results); first() just carries that through unchanged.
    n_ici   = dplyr::first(n_ici),
    n_other = dplyr::first(n_other),
    .groups = "drop"
  )

SECTION_LEVELS <- c("ICI Specificity", "ICI Therapy", "Non-ICI Therapy", "Chemotherapy",
                    "Targeted therapy", "Hormonal therapy")

# The five interaction (ICI-specificity) tests are grouped into their own
# leading panel rather than embedded at the top of each comparator's panel.
# They're a different quantity (a ratio of ORs/HRs between two groups, not a
# within-group effect size), and grouping them together lets the five
# specificity tests be read side by side as the figure's headline result,
# with the per-group panels below providing the supporting detail.
GROUP_ORDER <- c(
  "Non-ICI Therapy__interaction",
  "Non-ICI (MSS)__interaction",
  "Chemotherapy__interaction",
  "Targeted therapy__interaction",
  "Hormonal therapy__interaction",
  "ICI Therapy",
  "Non-ICI Therapy",
  "Chemotherapy",
  "Targeted therapy",
  "Hormonal therapy"
)

label_df <- group_n |>
  dplyr::mutate(
    # Interaction rows: name the comparator explicitly. The section strip
    # text is hidden in this theme (strip.text.y is blank below), so the row
    # label is the only place this appears — it must be self-explanatory on
    # its own, not just "Interaction" with the comparator implied by context.
    # "Non-ICI (MSS)" -> "Non-ICI, MSS" (comma instead of parenthetical) is a
    # one-off relabel specific to that comparator; every other comparator
    # only goes through the " Therapy" suffix strip.
    display_name = dplyr::case_when(
      group_level == "interaction" ~ paste0(
        "ICI vs. ",
        gsub(" \\(MSS\\)$", ", MSS", sub(" [Tt]herapy$", "", parent))
      ),
      TRUE ~ group
    ),
    # Interaction rows show both arms' sizes directly as "N = X vs. Y" --
    # order matches "ICI vs. <comparator>" in display_name, so the first
    # number is always the ICI arm and the second the comparator, without
    # repeating "ICI"/the comparator's name a second time. big.mark = ","
    # formats thousands (e.g. 2,432) for readability.
    # trim = TRUE matters here: format() on a whole column (vector) pads
    # every value to the width of the widest one for alignment -- fine for
    # a printed table, but it leaves a stray leading space in a string like
    # "ICI Therapy (N =  647)" when another row's N (e.g. 2,432) is wider.
    # Same fix already used for cohort N-labels in 0_landscape.ipynb /
    # 7_predicted_response_landscape.ipynb.
    group_label  = dplyr::case_when(
      group_level == "interaction" ~ paste0(
        display_name, " (N = ", format(n_ici, big.mark = ",", trim = TRUE),
        " vs. ", format(n_other, big.mark = ",", trim = TRUE), ")"
      ),
      TRUE ~ paste0(display_name, " (N = ", format(n_total, big.mark = ",", trim = TRUE), ")")
    )
  )

label_order <- label_df |>
  dplyr::filter(group %in% GROUP_ORDER) |>
  dplyr::mutate(row_order = match(group, GROUP_ORDER)) |>
  dplyr::arrange(row_order) |>
  dplyr::pull(group_label) |>
  rev()

ENDPOINT_LEVELS <- c("BOR", "DCB", "PFS", "OS")

# ── Build plot data ───────────────────────────────────────────────────────────

forest_df <- results |>
  dplyr::left_join(
    label_df |> dplyr::select(group, group_level, group_label),
    by = c("group", "group_level")
  ) |>
  dplyr::mutate(
    group_label = factor(group_label, levels = label_order),
    endpoint    = factor(endpoint, levels = ENDPOINT_LEVELS),
    # Row facet: one panel per treatment group; space = "free_y" sizes
    # each panel proportionally to its row count.
    section     = factor(
      dplyr::case_when(
        # All five interaction tests live in their own leading panel,
        # regardless of which comparator they test against.
        group_level == "interaction"  ~ "ICI Specificity",
        group == "ICI Therapy"        ~ "ICI Therapy",
        group == "Non-ICI Therapy"    ~ "Non-ICI Therapy",
        group == "Chemotherapy"       ~ "Chemotherapy",
        group == "Targeted therapy"   ~ "Targeted therapy",
        TRUE                           ~ "Hormonal therapy"
      ),
      levels = SECTION_LEVELS
    ),
    pt_alpha  = ifelse(!is.na(p.adj) & p.adj < 0.1, 1.0, 0.5),
    sig_star  = dplyr::case_when(
      !is.na(p.adj) & p.adj < 0.001 ~ "***",
      !is.na(p.adj) & p.adj < 0.01  ~ "**",
      !is.na(p.adj) & p.adj < 0.1   ~ "*",
      TRUE                          ~ ""
    ),
    # row_type drives point shape: ICI rows, Non-ICI rows, and the
    # ICI-vs-comparator interaction rows (drawn as diamonds, see below) each
    # get their own shape so the interaction term is never mistaken for a
    # regular within-group effect size.
    row_type = factor(
      dplyr::case_when(
        group_level == "interaction" ~ "Interaction",
        section == "ICI Therapy"     ~ "ICI",
        TRUE                          ~ "Non-ICI"
      ),
      levels = c("Interaction", "ICI", "Non-ICI")
    )
  )

# ── Debug check ───────────────────────────────────────────────────────────────
# Lightweight sanity check only (no debug figure, unlike
# 15_ici_specificity.ipynb) -- this notebook fits far fewer groups, so a
# missing-label warning here is enough to catch a GROUP_ORDER typo.

na_groups <- forest_df |> dplyr::filter(is.na(group_label)) |> dplyr::distinct(group)
if (nrow(na_groups) > 0) {
  warning("Groups with NA label (not in GROUP_ORDER): ",
          paste(na_groups$group, collapse = ", "))
}

plot_df <- forest_df |> dplyr::filter(!is.na(group_label))

# ── Main forest plot ──────────────────────────────────────────────────────────

adaptive_breaks <- function(x) {
  if (max(x, na.rm = TRUE) > 2) c(0.5, 1, 2) else c(0.7, 1, 1.3)
}
# Adaptive x-axis breaks: wide for BOR/DCB (OR), tight for PFS/OS (HR)

p_ici_specificity_main <- ggplot2::ggplot(
  plot_df,
  ggplot2::aes(
    x      = estimate,
    y      = group_label,
    alpha  = pt_alpha,
    shape  = row_type
  )
) +
  # Faint shading to visually separate ICI Therapy panels from the rest
  ggplot2::geom_rect(
    data = data.frame(
      section  = factor("ICI Therapy", levels = SECTION_LEVELS),
      endpoint = factor(c("BOR", "DCB", "PFS", "OS"), levels = ENDPOINT_LEVELS)
    ),
    ggplot2::aes(xmin = -Inf, xmax = Inf, ymin = -Inf, ymax = Inf),
    fill = COL_POSITIVE, alpha = 0.08,
    inherit.aes = FALSE
  ) +
  ggplot2::geom_vline(xintercept = 1, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.3) +
  # Regular per-group effect-size rows (ICI / Non-ICI circles + triangles)
  ggplot2::geom_pointrange(
    data = dplyr::filter(plot_df, group_level != "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    colour = COL_NEGATIVE, fatten = 3, size = 0.5
  ) +
  # Interaction rows (ICI-specificity test): drawn separately, in an accent
  # colour and as diamonds, so they read as a distinct quantity (a ratio of
  # ORs/HRs between groups) rather than another per-group effect size.
  ggplot2::geom_pointrange(
    data = dplyr::filter(plot_df, group_level == "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    colour = COL_ACCENT, fatten = 3.5, size = 0.6
  ) +
  ggplot2::geom_text(
    ggplot2::aes(x = conf.high, label = sig_star),
    hjust = -0.2, size = 2.8, colour = COL_INK, show.legend = FALSE
  ) +
  ggplot2::scale_shape_manual(
    values = c("Interaction" = 18, "ICI" = 16, "Non-ICI" = 17),
    # row_type is an explicit factor (levels = c("Interaction", "ICI",
    # "Non-ICI")) precisely so guide_legend()'s default top-to-bottom
    # ordering (first factor level on top) shows Interaction first.
    name = NULL
  ) +
  # section (rows) × endpoint (columns) — panel height scales with row count
  ggplot2::facet_grid(
    section ~ endpoint,
    scales = "free",
    space  = "free_y",
    labeller = labeller(
      section = c(
        "ICI Specificity"  = "Interaction",
        "ICI Therapy"      = "ICI",
        "Non-ICI Therapy"  = "Non-ICI",
        "Chemotherapy"     = "Chemo",
        "Targeted therapy" = "Targeted",
        "Hormonal therapy" = "Hormonal"
      ),
      endpoint = c(
        "BOR" = paste0("<b style='font-size:10pt'>BOR</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Worse ←  → Better</span>"),
        "DCB" = paste0("<b style='font-size:10pt'>DCB</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Worse ←  → Better</span>"),
        "PFS" = paste0("<b style='font-size:10pt'>PFS</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Better ←  → Worse</span>"),
        "OS"  = paste0("<b style='font-size:10pt'>OS</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Better ←  → Worse</span>")
      )
    )
  ) +
  ggplot2::scale_alpha_identity() +
  ggplot2::scale_x_log10(breaks = adaptive_breaks,
                         labels = scales::label_number(accuracy = 0.1, drop0trailing = TRUE)) +
  go_theme +
  ggplot2::theme(
    panel.grid.major.y = ggplot2::element_blank(),
    panel.grid.minor.x = ggplot2::element_blank(),
    # Column strips (endpoints)
    strip.text.x        = ggtext::element_markdown(size = 8, halign = 0.5),
    # Row strips (ICI / Non-ICI) — horizontal text on right margin
    strip.text.y       = ggplot2::element_blank(),
    strip.background.y = ggplot2::element_blank(),
    panel.spacing.y    = ggplot2::unit(0.69, "lines")
  ) +
  ggplot2::labs(
    x        = "OR / HR (95% CI, log scale)",
    y        = NULL,
    title    = "CIDER T-cell Score: ICI-Specific Signal, Adjusted for TMB",
    caption  = paste(
      "Adjusted for tumor type, age, sex, grouped biopsy location, tumor purity, and TMB.",
      "Intervals: 95% Wald CI.  Stars: * FDR < 0.1  ** FDR < 0.01  *** FDR < 0.001 (BH-adjusted across all tests within each endpoint).",
      "Diamonds: ICI-vs-comparator interaction term (ratio of ORs/HRs) — tests whether the CIDER effect differs by treatment.",
      "Simplified, TMB-adjusted main-figure view — see 15_ici_specificity.ipynb for the full mechanism-level breakdown and",
      "MSI-high/TMB-stratified sensitivity tests (not TMB-adjusted).",
      sep = "\n"
    )
  )

n_rows <- dplyr::n_distinct(plot_df$group_label, na.rm = TRUE)
fig_h  <- max(3.5, n_rows * 0.42 + 1.4)
set_plot_dims(10, fig_h); p_ici_specificity_main

saveRDS(p_ici_specificity_main, paste0(CIDER_PROJ_DIR, "figs/main_ici_specificity_figure.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/main_ici_specificity_figure.png"),
       p_ici_specificity_main, width = 10, height = fig_h, dpi = 300)
message("Saved main_ici_specificity_figure.png")


## 6 — Alternative figure: covariate-adjustment sensitivity (shape-encoded)

**Diagnostic / alternative view, not the submission figure.** Refits every
model in this notebook -- the five per-group effect sizes (ICI Therapy,
Non-ICI Therapy, and the three treatment classes) and the five
ICI-specificity interaction tests -- under three covariate-adjustment specs,
and plots all three on one forest plot with **adjustment spec encoded as
point shape**, dodged within each row:

- **Unadjusted** -- `cider_log2` (or `cider_log2 * is_ici` for the
  interaction rows) alone, no covariates at all (same convention as
  `5_models.ipynb`'s own "None" spec).
- **Adjusted** -- this notebook's primary spec: cohort_pan + age + sex +
  biopsy location + purity + TMB (the exact model `main_ici_specificity_figure`
  is built from, Section 5).
- **Adjusted (no purity)** -- the same minus `scaled_purity` (Section 4b's
  sensitivity check, extended here to the per-group rows too and shown
  visually with CIs rather than only as a printed table).

The fitted population is identical across all three specs for a given
group/endpoint -- `drop_na()` in every `fit_*()` function always requires
every adjustment column regardless of which spec is being fit, so any
difference between the three dodged points is attributable to the
covariate set alone, not a different sample. Interaction rows keep the
accent colour and per-group rows keep grey (same convention as the primary
figure) -- shape is reserved for adjustment spec only here.

Significance stars use their own Benjamini-Hochberg correction, computed
within this figure's own test family (3 specs × 11 groups × 4 endpoints) --
a separate family from, and not numerically comparable to, `results`' FDR
correction above (this notebook's primary, single-spec family).

**Output:** `figs/main_ici_specificity_figure_adjustment_sensitivity.rds`,
`figs/main_ici_specificity_figure_adjustment_sensitivity.png` -- does not
overwrite or change `main_ici_specificity_figure` (Section 5).


In [ ]:
# ── Fit all groups under all three adjustment specs ──────────────────────────

alt_parent_results <- purrr::map_dfr(
  ADJUSTMENT_LEVELS,
  function(adj) {
    purrr::imap(parent_dfs, ~ run_all_endpoints(.x, .y, adjustment = adj)) |>
      dplyr::bind_rows()
  }
) |>
  dplyr::mutate(
    group_level = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), "top", "mid"),
    parent      = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), NA_character_, "Non-ICI Therapy")
  )

alt_interaction_results <- purrr::map_dfr(
  ADJUSTMENT_LEVELS,
  function(adj) {
    purrr::imap(interaction_dfs, ~ run_interaction_endpoints(.x, .y, adjustment = adj)) |>
      dplyr::bind_rows()
  }
) |>
  dplyr::mutate(
    group_level = "interaction",
    parent      = sub("__interaction$", "", group)
  ) |>
  dplyr::left_join(interaction_n_df, by = "group")

alt_results <- dplyr::bind_rows(alt_parent_results, alt_interaction_results) |>
  dplyr::mutate(adjustment = factor(adjustment, levels = ADJUSTMENT_LEVELS)) |>
  # Own FDR family for this figure (see markdown above) -- 3x the tests per
  # endpoint vs. the primary `results` family, so deliberately kept separate.
  dplyr::group_by(endpoint) |>
  dplyr::mutate(p.adj = p.adjust(p.value, method = "fdr")) |>
  dplyr::ungroup()

message("\nAdjustment-sensitivity models fitted: ", nrow(alt_results), " rows (",
        dplyr::n_distinct(alt_results$group), " groups × ", length(ADJUSTMENT_LEVELS),
        " adjustment specs × 4 endpoints, allowing for skipped small-N fits)")

# ── Build plot data ───────────────────────────────────────────────────────────
# Reuses label_df / label_order / SECTION_LEVELS / ENDPOINT_LEVELS / GROUP_ORDER
# / adaptive_breaks() from Section 5 above -- N per group doesn't depend on
# adjustment spec (drop_na() requires the same columns regardless), so the
# existing labels (built from the primary `results`) apply unchanged here.

alt_forest_df <- alt_results |>
  dplyr::left_join(
    label_df |> dplyr::select(group, group_level, group_label),
    by = c("group", "group_level")
  ) |>
  dplyr::mutate(
    group_label = factor(group_label, levels = label_order),
    endpoint    = factor(endpoint, levels = ENDPOINT_LEVELS),
    section     = factor(
      dplyr::case_when(
        group_level == "interaction"  ~ "ICI Specificity",
        group == "ICI Therapy"        ~ "ICI Therapy",
        group == "Non-ICI Therapy"    ~ "Non-ICI Therapy",
        group == "Chemotherapy"       ~ "Chemotherapy",
        group == "Targeted therapy"   ~ "Targeted therapy",
        TRUE                           ~ "Hormonal therapy"
      ),
      levels = SECTION_LEVELS
    ),
    pt_alpha  = ifelse(!is.na(p.adj) & p.adj < 0.1, 1.0, 0.5),
    sig_star  = dplyr::case_when(
      !is.na(p.adj) & p.adj < 0.001 ~ "***",
      !is.na(p.adj) & p.adj < 0.01  ~ "**",
      !is.na(p.adj) & p.adj < 0.1   ~ "*",
      TRUE                          ~ ""
    )
  ) |>
  dplyr::filter(!is.na(group_label))

ADJUSTMENT_DODGE <- ggplot2::position_dodge(width = 0.6)

p_ici_specificity_adjustment_sensitivity <- ggplot2::ggplot(
  alt_forest_df,
  ggplot2::aes(
    x      = estimate,
    y      = group_label,
    alpha  = pt_alpha,
    shape  = adjustment
  )
) +
  ggplot2::geom_rect(
    data = data.frame(
      section  = factor("ICI Therapy", levels = SECTION_LEVELS),
      endpoint = factor(c("BOR", "DCB", "PFS", "OS"), levels = ENDPOINT_LEVELS)
    ),
    ggplot2::aes(xmin = -Inf, xmax = Inf, ymin = -Inf, ymax = Inf),
    fill = COL_POSITIVE, alpha = 0.08,
    inherit.aes = FALSE
  ) +
  ggplot2::geom_vline(xintercept = 1, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.3) +
  # Per-group rows: grey, same as the primary figure -- colour is reserved
  # for the interaction-vs-per-group distinction, shape for adjustment spec.
  ggplot2::geom_pointrange(
    data = dplyr::filter(alt_forest_df, group_level != "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    colour = COL_NEGATIVE, fatten = 2.6, size = 0.45,
    position = ADJUSTMENT_DODGE
  ) +
  # Interaction rows: accent colour, same as the primary figure.
  ggplot2::geom_pointrange(
    data = dplyr::filter(alt_forest_df, group_level == "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    colour = COL_ACCENT, fatten = 3.0, size = 0.55,
    position = ADJUSTMENT_DODGE
  ) +
  ggplot2::geom_text(
    ggplot2::aes(x = conf.high, label = sig_star),
    position = ADJUSTMENT_DODGE,
    hjust = -0.2, size = 2.4, colour = COL_INK, show.legend = FALSE
  ) +
  ggplot2::scale_shape_manual(
    values = c("Unadjusted" = 1, "Adjusted" = 16, "Adjusted (no purity)" = 17),
    name   = "Covariates"
  ) +
  ggplot2::facet_grid(
    section ~ endpoint,
    scales = "free",
    space  = "free_y",
    labeller = labeller(
      section = c(
        "ICI Specificity"  = "Interaction",
        "ICI Therapy"      = "ICI",
        "Non-ICI Therapy"  = "Non-ICI",
        "Chemotherapy"     = "Chemo",
        "Targeted therapy" = "Targeted",
        "Hormonal therapy" = "Hormonal"
      ),
      endpoint = c(
        "BOR" = paste0("<b style='font-size:10pt'>BOR</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Worse ←  → Better</span>"),
        "DCB" = paste0("<b style='font-size:10pt'>DCB</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Worse ←  → Better</span>"),
        "PFS" = paste0("<b style='font-size:10pt'>PFS</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Better ←  → Worse</span>"),
        "OS"  = paste0("<b style='font-size:10pt'>OS</b><br><span style='font-size:7pt; color:", COL_NEGATIVE, "'>Better ←  → Worse</span>")
      )
    )
  ) +
  ggplot2::scale_alpha_identity() +
  ggplot2::scale_x_log10(breaks = adaptive_breaks,
                         labels = scales::label_number(accuracy = 0.1, drop0trailing = TRUE)) +
  go_theme +
  ggplot2::theme(
    panel.grid.major.y = ggplot2::element_blank(),
    panel.grid.minor.x = ggplot2::element_blank(),
    strip.text.x        = ggtext::element_markdown(size = 8, halign = 0.5),
    strip.text.y       = ggplot2::element_blank(),
    strip.background.y = ggplot2::element_blank(),
    panel.spacing.y    = ggplot2::unit(0.9, "lines"),
    legend.position    = "bottom"
  ) +
  ggplot2::labs(
    x        = "OR / HR (95% CI, log scale)",
    y        = NULL,
    title    = "CIDER T-cell Score: ICI Specificity Under Three Covariate-Adjustment Specs",
    caption  = paste(
      "Shape = covariate-adjustment spec (see Section 6 intro). Same fitted population across all three specs per row.",
      "Stars: * FDR < 0.1  ** FDR < 0.01  *** FDR < 0.001, BH-adjusted within this figure's own test family",
      "(3 specs × 11 groups × 4 endpoints) -- a different family from, and not comparable to, Figure 6's stars.",
      "Diagnostic/alternative view of the Section 5 model -- see Section 7 below for the submission Figure 6.",
      sep = "\n"
    )
  )

n_rows_alt <- dplyr::n_distinct(alt_forest_df$group_label, na.rm = TRUE)
fig_h_alt  <- max(4.5, n_rows_alt * 0.6 + 1.8)
set_plot_dims(11, fig_h_alt); p_ici_specificity_adjustment_sensitivity

saveRDS(p_ici_specificity_adjustment_sensitivity,
        paste0(CIDER_PROJ_DIR, "figs/main_ici_specificity_figure_adjustment_sensitivity.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/main_ici_specificity_figure_adjustment_sensitivity.png"),
       p_ici_specificity_adjustment_sensitivity, width = 11, height = fig_h_alt, dpi = 300)
message("Saved main_ici_specificity_figure_adjustment_sensitivity.png")


## 7 — Figure 6: predictive (ICI) vs prognostic (cohort-specific) effects

**This is the submission figure (Figure 6).** Originally built as a
diagnostic alternative to Section 5/6's design, then adopted as the
main-text figure in place of Section 5's own forest plot (the former Figure
7, now retired -- Section 5's code is kept only because this section reuses
its shared fitting/labeling helpers). Simplified companion to Section 6,
per direction:

- Only the two covariate specs `5_models.ipynb`/`8_forest.ipynb` use
  everywhere else -- **None** and **TMB/Clinical/Purity** -- not Section 6's
  three (drops "Adjusted (no purity)"). Same shapes as the forest-plot
  supplement: circle (16) for None, triangle (17) for TMB/Clinical/Purity.
- Row grouping collapses from one panel per treatment group (Section 5/6's
  six `SECTION_LEVELS`) down to just two, **Cohort-Specific Effect** on top
  and **Interaction / Relative Effect** below, per direction:
    - **Cohort-Specific Effect** -- each group's own within-group effect
      size (ICI Therapy, Non-ICI Therapy, Chemotherapy, Targeted therapy,
      Hormonal therapy), i.e. whether CIDER has value regardless of
      treatment (prognostic).
    - **Interaction / Relative Effect** -- the four ICI-specificity
      interaction tests (whether CIDER's effect differs between ICI and a
      comparator, i.e. predictive). Drops the "Non-ICI (MSS)" test that
      Section 5/6 include -- per direction, MSS sensitivity checks (this
      one plus two new MSS-restricted main-effect rows) now live in
      15_ici_specificity.ipynb instead, keeping this figure scoped to the
      headline comparisons.
- Only BOR and OS (drops DCB and PFS) -- per direction, for simplicity in
  this main figure. Endpoint facet headers are expanded to full names,
  "Best Overall Response (BOR)" / "Overall Survival (OS)".
- Thinner and shorter than Section 6: narrower (2 specs instead of 3, 2
  endpoints instead of 4) and much shorter (2 row-groups instead of 6).
- Caption trimmed to just the significance-star legend -- the shape/
  covariate and interaction/cohort-specific explanations are dropped since
  the row-section grouping and legend now carry that meaning directly.
- **One `facet_grid(section ~ endpoint, scales = "free", space = "free_y")`
  plot, no section headers.** An earlier version tried styling the row
  strip as a banner-style divider between the two sections, which doesn't
  work (a facet_grid row strip is a side gutter next to the panel, not a
  banner spanning the endpoint columns); a later version instead built two
  separate ggplot objects (each with its own bold plot title) stacked via
  patchwork, which worked visually but reintroduced the same plumbing as
  Section 5/6's `guides = "collect"` legend-merging (fragile across
  ggplot2/patchwork versions, as hit with the `&` operator). Both section
  headers are now dropped entirely in favour of one single plot
  (`strip.text.y`/`strip.background.y` blanked) with a visible vertical gap
  (`panel.spacing.y`) between the Cohort-Specific Effect rows and the
  Interaction / Relative Effect rows -- one shared legend, no patchwork
  involved.
- Small italic labels identify each block without a full banner header:
  "Interaction / Relative Effects" in the gap *between* the two row-facets,
  and "Cohort-Specific Effects" in a matching gap added *above* the first
  row-facet. ggplot has no layer that draws inside `panel.spacing` -- it's
  gtable whitespace outside any panel's coordinate space -- so both labels
  are inserted as `textGrob`s directly into the rendered gtable: the lower
  one into facet_grid's own spacer row (the row strictly between the two
  row-facets' panel rows, already created by the `panel.spacing.y` theme
  setting), the upper one into a brand-new row spliced in via
  `gtable::gtable_add_rows()`, since there's no facet-generated spacer
  above the first row-facet to repurpose. Both span the full panel width.
  This means the saved object (`ici_specificity_predictive_vs_prognostic.rds`)
  is a gtable/grob, not a plain ggplot object, from this point on --
  `assemble_main_figures.R` wraps it into `figure_6.png` via
  `wrap_elements()`, which accepts a grob the same as any ggplot.

Own FDR family (2 specs x 9 groups x 2 endpoints) -- different from both
Section 5's primary family (Section 4) and Section 6's 3-spec, 4-endpoint
family, so its stars aren't directly comparable to either.


In [ ]:
# ── Fit all groups under the two 5_models.ipynb/8_forest.ipynb covariate specs ──
# Internal adjustment names stay "Unadjusted"/"Adjusted" (what
# build_adjustment_covars() keys off); relabeled to "None"/"TMB/Clinical/
# Purity" below for display, matching 5_models.ipynb/8_forest.ipynb exactly.
PREDPROG_ADJUSTMENT_LEVELS <- c("Unadjusted", "Adjusted")

predprog_parent_results <- purrr::map_dfr(
  PREDPROG_ADJUSTMENT_LEVELS,
  function(adj) {
    purrr::imap(parent_dfs, ~ run_all_endpoints(.x, .y, adjustment = adj)) |>
      dplyr::bind_rows()
  }
) |>
  dplyr::mutate(
    group_level = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), "top", "mid"),
    parent      = ifelse(group %in% c("ICI Therapy", "Non-ICI Therapy"), NA_character_, "Non-ICI Therapy")
  )

# Note: the MSS-restricted sensitivity rows (ICI Therapy/Non-ICI Therapy's
# main effects restricted to microsatellite-stable patients) live in
# 15_ici_specificity.ipynb instead of here -- this figure stays scoped to
# the headline predictive/prognostic comparisons; MSS robustness checks sit
# with the other stratified sensitivity analyses (MSI-high, TMB-High/Low)
# in the full supplement.

predprog_interaction_results <- purrr::map_dfr(
  PREDPROG_ADJUSTMENT_LEVELS,
  function(adj) {
    purrr::imap(interaction_dfs, ~ run_interaction_endpoints(.x, .y, adjustment = adj)) |>
      dplyr::bind_rows()
  }
) |>
  dplyr::mutate(
    group_level = "interaction",
    parent      = sub("__interaction$", "", group)
  ) |>
  dplyr::left_join(interaction_n_df, by = "group")

predprog_results <- dplyr::bind_rows(predprog_parent_results, predprog_interaction_results) |>
  dplyr::mutate(
    # Level order here drives position_dodge()'s vertical stacking below
    # (first level dodges to the bottom of each group's slot, last level to
    # the top) -- "TMB/Clinical/Purity" first / "None" last puts the
    # Unadjusted bar above the Adjusted bar for every group, per direction.
    # This is independent of the legend's display order, set separately via
    # scale_shape_manual(breaks = ...) below.
    adjustment = factor(
      dplyr::recode(adjustment, "Unadjusted" = "None", "Adjusted" = "TMB/Clinical/Purity"),
      levels = c("TMB/Clinical/Purity", "None")
    )
  ) |>
  # Own FDR family for this figure -- see markdown above.
  dplyr::group_by(endpoint) |>
  dplyr::mutate(p.adj = p.adjust(p.value, method = "fdr")) |>
  dplyr::ungroup()

message("Predictive-vs-prognostic models fitted: ", nrow(predprog_results), " rows (",
        dplyr::n_distinct(predprog_results$group), " groups x ", length(PREDPROG_ADJUSTMENT_LEVELS),
        " adjustment specs x 4 endpoints, allowing for skipped small-N fits)")


In [ ]:
# ── Build plot data: 2 row groups instead of 6 ────────────────────────────────
# Reuses label_df / ENDPOINT_LEVELS / adaptive_breaks() from Section 5. Row
# order is local to this figure (PREDPROG_GROUP_ORDER), not the shared
# GROUP_ORDER, since this figure drops the "Non-ICI (MSS)" interaction test
# (moved to 15_ici_specificity.ipynb -- see note in the fitting cell above).
# Cohort-Specific Effect (overall/per-group main effects) on top, Interaction / Relative Effect (comparative tests) below -- per direction.
PREDPROG_SECTION_LEVELS <- c("Cohort-Specific Effect", "Interaction / Relative Effect")

predprog_label_df <- label_df

PREDPROG_GROUP_ORDER <- c(
  "ICI Therapy", "Non-ICI Therapy", "Chemotherapy", "Targeted therapy", "Hormonal therapy",
  "Non-ICI Therapy__interaction",
  "Chemotherapy__interaction", "Targeted therapy__interaction", "Hormonal therapy__interaction"
)

predprog_label_order <- predprog_label_df |>
  dplyr::filter(group %in% PREDPROG_GROUP_ORDER) |>
  dplyr::mutate(row_order = match(group, PREDPROG_GROUP_ORDER)) |>
  dplyr::arrange(row_order) |>
  dplyr::pull(group_label) |>
  rev()

# Main-figure simplicity: only BOR and OS (drop DCB and PFS) -- local
# endpoint set for this figure only, not the shared ENDPOINT_LEVELS.
PREDPROG_ENDPOINT_LEVELS <- c("BOR", "OS")

predprog_forest_df <- predprog_results |>
  dplyr::filter(endpoint %in% PREDPROG_ENDPOINT_LEVELS) |>
  dplyr::left_join(
    predprog_label_df |> dplyr::select(group, group_level, group_label),
    by = c("group", "group_level")
  ) |>
  dplyr::mutate(
    group_label = factor(group_label, levels = predprog_label_order),
    endpoint    = factor(endpoint, levels = PREDPROG_ENDPOINT_LEVELS),
    # Just two row-facets: every interaction test together, every
    # per-group (cohort-specific) effect together -- no more one panel per
    # treatment group.
    section     = factor(
      ifelse(group_level == "interaction", "Interaction / Relative Effect", "Cohort-Specific Effect"),
      levels = PREDPROG_SECTION_LEVELS
    ),
    pt_alpha  = ifelse(!is.na(p.adj) & p.adj < 0.1, 1.0, 0.5),
    sig_star  = dplyr::case_when(
      !is.na(p.adj) & p.adj < 0.001 ~ "***",
      !is.na(p.adj) & p.adj < 0.01  ~ "**",
      !is.na(p.adj) & p.adj < 0.1   ~ "*",
      TRUE                          ~ ""
    )
  ) |>
  dplyr::filter(!is.na(group_label))

# Numeric y position instead of relying on group_label's plain discrete
# factor: lets a little *extra* space be inserted above the ICI Therapy row
# specifically, separating it from the other within-group cohorts
# (Non-ICI/Chemo/Targeted/Hormonal), without disturbing the uniform
# spacing used everywhere else. Position is sequential (1, 2, 3, ...) per
# section, in the same bottom-to-top display order group_label's factor
# levels already encode, plus +0.25 (a quarter of the normal 1-unit gap)
# added only above "ICI Therapy", the top-most row of the Cohort-Specific
# Effect section. facet_grid(scales = "free") computes each row-facet
# panel's range independently, so the two sections' positions are free to
# reuse the same small integers -- except scale_y_continuous()'s breaks/
# labels aren't facet-aware, so Interaction / Relative Effect's positions
# are offset by a large constant purely to keep the two sections' ranges
# from overlapping; each panel only ever draws the breaks inside its own
# range, so this offset is invisible in the rendered figure.
PREDPROG_SECTION_Y_OFFSET <- c("Cohort-Specific Effect" = 0, "Interaction / Relative Effect" = 100)

predprog_y_lookup <- predprog_forest_df |>
  dplyr::distinct(section, group, group_label) |>
  dplyr::group_by(section) |>
  dplyr::arrange(group_label, .by_group = TRUE) |>
  dplyr::mutate(y_pos = seq_len(dplyr::n()) + ifelse(group == "ICI Therapy", 0.25, 0)) |>
  dplyr::ungroup() |>
  dplyr::mutate(y_pos = y_pos + PREDPROG_SECTION_Y_OFFSET[as.character(section)])

predprog_forest_df <- predprog_forest_df |>
  dplyr::left_join(predprog_y_lookup |> dplyr::select(section, group, y_pos), by = c("section", "group"))

# orientation = "y" makes the dodge direction explicit -- needed now that y
# is a plain numeric column rather than a factor ggplot could auto-detect
# as the categorical/dodge axis.
PREDPROG_DODGE <- ggplot2::position_dodge(width = 0.5, orientation = "y")

# Symmetric x-axis around 1 (the null effect) per facet panel, on the log
# scale -- i.e. the panel's lower and upper bound multiply to 1, so 1 sits
# exactly in the visual centre instead of wherever the data's own min/max CI
# happen to fall. facet_grid(scales = "free") sizes each panel from the
# data it's given, so the trick is to extend that data per (section,
# endpoint) with two invisible phantom points at the symmetric bound via
# geom_blank() -- they contribute to the panel's axis range without drawing
# anything. 5% padding beyond the widest real CI keeps points/stars off the
# panel edge.
predprog_xrange_df <- predprog_forest_df |>
  dplyr::filter(is.finite(conf.low), is.finite(conf.high), conf.low > 0, conf.high > 0) |>
  dplyr::group_by(section, endpoint) |>
  dplyr::summarise(
    log_bound = max(abs(log10(conf.low)), abs(log10(conf.high))) * 1.05,
    .groups = "drop"
  )
predprog_xrange_df <- dplyr::bind_rows(
  predprog_xrange_df |> dplyr::mutate(estimate =  10 ^ log_bound),
  predprog_xrange_df |> dplyr::mutate(estimate = 10 ^ -log_bound)
)

# ── Single forest plot, one row-facet per section ─────────────────────────────
# Back to one facet_grid(section ~ endpoint) plot rather than two patchwork-
# stacked panels: no bold section headers this time (removed per direction),
# just a visible vertical gap between the Cohort-Specific Effect rows and
# the Interaction / Relative Effect rows, via panel.spacing.y below -- with
# strip.text.y/strip.background.y blanked out (same device Section 5/6 use
# for their own row strips), the facet boundary reads as pure whitespace, not
# a labelled divider. This also sidesteps every patchwork guide-collection
# headache from the two-panel version: one ggplot means one legend, with no
# need for `&`, guides = "collect", or guide_area().
p_ici_predictive_vs_prognostic <- ggplot2::ggplot(
  predprog_forest_df,
  ggplot2::aes(x = estimate, y = y_pos, alpha = pt_alpha, shape = adjustment, colour = section)
) +
  ggplot2::geom_vline(xintercept = 1, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.3) +
  # Invisible points that force each (section, endpoint) panel's x-range to
  # be symmetric around 1 -- see predprog_xrange_df above.
  ggplot2::geom_blank(data = predprog_xrange_df, ggplot2::aes(x = estimate), inherit.aes = FALSE) +
  # Cohort-Specific Effect rows: grey, same convention as Section 5/6.
  ggplot2::geom_pointrange(
    data = dplyr::filter(predprog_forest_df, group_level != "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    fatten = 2.6, size = 0.42,
    position = PREDPROG_DODGE
  ) +
  # Interaction / Relative Effect rows: accent colour, same as Section 5/6.
  ggplot2::geom_pointrange(
    data = dplyr::filter(predprog_forest_df, group_level == "interaction"),
    ggplot2::aes(xmin = conf.low, xmax = conf.high),
    fatten = 3.0, size = 0.5,
    position = PREDPROG_DODGE
  ) +
  ggplot2::geom_text(
    ggplot2::aes(x = conf.high, label = sig_star),
    position = PREDPROG_DODGE,
    hjust = -0.2, size = 2.6, colour = COL_INK, show.legend = FALSE
  ) +
  # Colour now mapped to section (rather than hardcoded per layer) so it
  # gets its own "Effect Type" legend, distinguishing Cohort-Specific Effect
  # (the baseline/reference per-group reading -> COL_NEGATIVE) from
  # Interaction / Relative Effect (the comparative interaction test -> the
  # general-purpose two-category accent, COL_ACCENT) -- same two colours as
  # before, just legend-visible now that the section headers that used to
  # convey this are gone.
  ggplot2::scale_colour_manual(
    values = c("Cohort-Specific Effect" = COL_NEGATIVE, "Interaction / Relative Effect" = COL_ACCENT),
    name   = "Effect Type"
  ) +
  # Same shapes as the forest-plot supplement (8_forest.ipynb): circle for
  # None, triangle for TMB/Clinical/Purity. breaks fixes the legend's
  # display order independent of the adjustment factor's level order
  # (reversed on purpose to control dodge stacking instead).
  ggplot2::scale_shape_manual(
    values = c("None" = 16, "TMB/Clinical/Purity" = 17),
    breaks = c("None", "TMB/Clinical/Purity"),
    name   = "Covariates"
  ) +
  # section (rows) x endpoint (columns) -- panel height scales with row
  # count; space = "free_y" is what keeps the Interaction block from being
  # stretched to match the taller Cohort-Specific block.
  ggplot2::facet_grid(
    section ~ endpoint,
    scales   = "free",
    space    = "free_y",
    labeller = labeller(endpoint = c(
      "BOR" = paste0("<b style='font-size:10pt'>Best Overall Response (BOR)</b><br><span style='font-size:8.5pt; color:", COL_NEGATIVE, "'>Worse ←  → Better</span>"),
      "OS"  = paste0("<b style='font-size:10pt'>Overall Survival (OS)</b><br><span style='font-size:8.5pt; color:", COL_NEGATIVE, "'>Better ←  → Worse</span>")
    ))
  ) +
  ggplot2::scale_alpha_identity() +
  ggplot2::scale_x_log10(breaks = adaptive_breaks,
                         labels = scales::label_number(accuracy = 0.1, drop0trailing = TRUE)) +
  # Breaks/labels come from the same lookup used to build y_pos above --
  # each facet panel only renders the subset of breaks that fall inside its
  # own (free-scaled) range, so this single global scale still shows the
  # correct labels per section despite the shared numbering.
  ggplot2::scale_y_continuous(
    breaks = predprog_y_lookup$y_pos,
    labels = as.character(predprog_y_lookup$group_label),
    expand = ggplot2::expansion(add = 0.6)
  ) +
  go_theme +
  ggplot2::theme(
    panel.grid.major.y  = ggplot2::element_blank(),
    panel.grid.minor.x  = ggplot2::element_blank(),
    strip.text.x        = ggtext::element_markdown(size = 8, halign = 0.5),
    # A little smaller than go_theme's default (11pt).
    axis.title.x        = ggplot2::element_text(size = 9.5),
    # No section headers -- blank the row strip entirely (text + background).
    # The gap between facet rows is widened below (panel.spacing.y) to fit
    # a small text label dropped directly into that whitespace band (see
    # the gtable step after this plot is built).
    strip.text.y        = ggplot2::element_blank(),
    strip.background.y  = ggplot2::element_blank(),
    panel.spacing.y      = ggplot2::unit(0.9, "lines"),
    legend.position      = "bottom",
    # Two distinct legends (Effect Type colour, Covariates shape) stacked
    # vertically rather than side by side, Effect Type on top -- matches
    # the stacking used elsewhere in this notebook's forest plots.
    # legend.box.just = "left" matters here: with legend.position = "bottom",
    # ggplot's default is to centre each stacked legend box independently
    # within the legend area, so "Effect Type" (fewer/narrower entries) and
    # "Covariates" (wider) end up with mismatched left edges. "left" aligns
    # both boxes -- and their titles -- to a shared left edge instead.
    legend.box           = "vertical",
    legend.box.just      = "left",
    # legend.spacing.y is what controls the gap *between* stacked legend
    # boxes when legend.box = "vertical" (legend.spacing alone only affects
    # the horizontal case) -- default leaves a noticeable blank band between
    # "Effect Type" and "Covariates"; tightened here so they read as one
    # compact legend block.
    legend.spacing.y     = ggplot2::unit(0, "pt"),
    legend.margin        = ggplot2::margin(t = 0, b = 0)
  ) +
  ggplot2::guides(
    colour = ggplot2::guide_legend(order = 1),
    shape  = ggplot2::guide_legend(order = 2)
  ) +
  ggplot2::labs(
    x       = "OR / HR (95% CI, log scale)",
    y       = NULL,
    title   = "Predictive Specificity for ICI Therapy vs. General Prognostic Effects",
    caption = "Stars: * FDR < 0.1  ** FDR < 0.01  *** FDR < 0.001 (BH-adjusted within this figure's own test family)."
  )

# ── Drop text labels into the gaps above/between the two row-sections ────────
# There's no ggplot layer that can draw inside panel.spacing -- it's pure
# gtable whitespace, outside any panel's coordinate space -- so this
# inserts text grobs directly into the rendered gtable. The gap *between*
# the two row-facets already exists (ggplot creates it from the
# panel.spacing.y theme setting above); facet_grid's gtable lists panel
# grobs as "panel-<row>-<col>", so every "Cohort-Specific Effect" panel
# (row 1, one per endpoint column) shares one gtable row index and every
# "Interaction / Relative Effect" panel (row 2) shares the next, with the
# spacer row strictly between them. Both labels span the full panel width
# (every endpoint column), not just one, so each reads as centred under the
# whole figure rather than under a single column.
predprog_grob <- ggplot2::ggplotGrob(p_ici_predictive_vs_prognostic)

panel_grobs    <- grep("^panel-", predprog_grob$layout$name)
panel_row_idx  <- sort(unique(predprog_grob$layout$t[panel_grobs]))
stopifnot(length(panel_row_idx) == 2)  # exactly 2 row-facets expected
gap_row_idx    <- panel_row_idx[1] + 1
panel_col_idx  <- range(predprog_grob$layout$l[panel_grobs])

predprog_grob <- gtable::gtable_add_grob(
  predprog_grob,
  grid::textGrob(
    "Interaction / Relative Effects",
    gp = grid::gpar(fontsize = 8, fontface = "italic", col = COL_NEGATIVE)
  ),
  t = gap_row_idx, b = gap_row_idx, l = panel_col_idx[1], r = panel_col_idx[2],
  name = "predprog-gap-label"
)

# Matching gap + label above the Cohort-Specific Effect block. Unlike the
# gap below it, there's no facet-generated spacer row here to fill --
# panel.spacing only inserts whitespace *between* facet panels, not above
# the first or below the last -- so a brand-new row has to be added to the
# gtable via gtable_add_rows() first. `pos` is 0-indexed from the top, so
# `panel_row_idx[1] - 1` places the new row immediately before the
# (original) top panel row; gtable_add_rows() shifts every row at or below
# that point down by one and updates $layout automatically, so the
# already-placed bottom-gap label above doesn't need to be touched. After
# the insertion, the new row's own index equals the top panel row's
# *original* index, since that's exactly where it was spliced in.
predprog_grob <- gtable::gtable_add_rows(
  predprog_grob, heights = ggplot2::unit(0.9, "lines"), pos = panel_row_idx[1] - 1
)
predprog_grob <- gtable::gtable_add_grob(
  predprog_grob,
  grid::textGrob(
    "Cohort-Specific Effects",
    gp = grid::gpar(fontsize = 8, fontface = "italic", col = COL_NEGATIVE)
  ),
  t = panel_row_idx[1], b = panel_row_idx[1], l = panel_col_idx[1], r = panel_col_idx[2],
  name = "predprog-top-gap-label"
)

n_rows_predprog <- dplyr::n_distinct(predprog_forest_df$group_label, na.rm = TRUE)
# Both gaps (above Cohort-Specific Effects, between the two sections) were
# halved from 1.8 to 0.9 lines -- trimmed the fixed offset accordingly.
fig_h_predprog  <- max(3, n_rows_predprog * 0.35 + 1.7)
set_plot_dims(7.7, fig_h_predprog)
grid::grid.newpage(); grid::grid.draw(predprog_grob)

saveRDS(predprog_grob, paste0(CIDER_PROJ_DIR, "figs/ici_specificity_predictive_vs_prognostic.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/ici_specificity_predictive_vs_prognostic.png"),
       predprog_grob, width = 7.7, height = fig_h_predprog, dpi = 300)
message("Saved ici_specificity_predictive_vs_prognostic.png")
